<a href="https://colab.research.google.com/github/abbbymuaa/mssp6070/blob/main/WeeklyModules/Week09/Week9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests

In [3]:
import requests
import os
import time

#ask the user to enter a photo category
category = input("Enter a photo category (e.g., cats, nature, flowers): ")

#create a folder for downloaded images
folder_name = category.replace(" ", "_") + "_images"
os.makedirs(folder_name, exist_ok=True)

#wikimedia Commons API
url = "https://commons.wikimedia.org/w/api.php"

#identify the program to Wikimedia
headers = {
    "User-Agent": "StudentImageDownloader/1.0 (Python learning project)"
}

#number of images to download
max_images = 20
downloaded = 0
search_continue = None

while downloaded < max_images:
    #search for images
    params = {
        "action": "query",
        "generator": "search",
        "gsrsearch": f"filetype:bitmap {category}",
        "gsrnamespace": 6,
        "gsrlimit": 20,
        "prop": "imageinfo",
        "iiprop": "url",
        "iiurlwidth": 800,
        "format": "json"
    }

    if search_continue:
        params.update(search_continue)

    try:
        response = requests.get(
            url, params=params, headers=headers, timeout=30
        )
        response.raise_for_status()
        data = response.json()

        pages = data.get("query", {}).get("pages", {})

        if not pages:
            print("No more images found.")
            break

        #download each image
        for page in pages.values():
            if downloaded >= max_images:
                break

            image_info = page.get("imageinfo", [])

            if not image_info:
                continue

            image_url = image_info[0].get("thumburl")

            if not image_url:
                image_url = image_info[0].get("url")

            if not image_url:
                continue

            try:
                image_response = requests.get(
                    image_url, headers=headers, timeout=30
                )
                image_response.raise_for_status()

                #save the image
                filename = f"image_{downloaded + 1}.jpg"
                file_path = os.path.join(folder_name, filename)

                with open(file_path, "wb") as image_file:
                    image_file.write(image_response.content)

                downloaded += 1
                print(f"Downloaded: {filename}")

                time.sleep(0.5)

            except requests.exceptions.RequestException as e:
                print("Failed to download an image:", e)

        #continue searching if more results exist
        search_continue = data.get("continue")

        if not search_continue:
            break

    except requests.exceptions.RequestException as e:
        print("Error:", e)
        break

print("\nDownload complete!")
print(f"Total images downloaded: {downloaded}")
print(f"Images saved in: {folder_name}")


Enter a photo category (e.g., cats, nature, flowers): cats
Downloaded: image_1.jpg
Downloaded: image_2.jpg
Downloaded: image_3.jpg
Downloaded: image_4.jpg
Downloaded: image_5.jpg
Downloaded: image_6.jpg
Downloaded: image_7.jpg
Downloaded: image_8.jpg
Downloaded: image_9.jpg
Downloaded: image_10.jpg
Downloaded: image_11.jpg
Downloaded: image_12.jpg
Downloaded: image_13.jpg
Downloaded: image_14.jpg
Downloaded: image_15.jpg
Downloaded: image_16.jpg
Downloaded: image_17.jpg
Downloaded: image_18.jpg
Downloaded: image_19.jpg
Downloaded: image_20.jpg

Download complete!
Total images downloaded: 20
Images saved in: cats_images
